# Captura Hantek — vista previa (matplotlib)

Lee `captura_scope.csv` generado con:

`python -m hantek_usb.cli get-real-data --parse --export-csv captura_scope.csv --count-a 0x400 --count-b 0`

Por defecto el CSV tiene **dos canales** (`ch1_u8`, `ch2_u8`): el firmware entrega **CH1, CH2, CH1, CH2…** (ver `PROTOCOLO_USB.md` §3.3). Si exportaste con `--no-interleaved`, el archivo tendrá una sola columna `adc_u8` y el gráfico mostrará una curva.

Las muestras son **ADC 8 bit** (0–255), sin conversión a voltios.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def load_hantek_scope_csv(path: Path):
    """CSV de --export-csv: interleaved (ch1_u8, ch2_u8) o raw (adc_u8 con --no-interleaved)."""
    header: list[str] | None = None
    rows: list[list[str]] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split(",")]
            if header is None:
                header = parts
                continue
            rows.append(parts)
    if header is None or not rows:
        raise ValueError("CSV vacío o sin datos")
    h = [x.lower() for x in header]
    def col(name: str) -> int:
        return h.index(name)

    data = np.asarray(rows, dtype=object)
    time_s = data[:, col("time_s")].astype(float)

    if "ch1_u8" in h and "ch2_u8" in h:
        ch1 = data[:, col("ch1_u8")].astype(int)
        ch2 = data[:, col("ch2_u8")].astype(int)
        return time_s, ch1, ch2, "interleaved"
    if "adc_u8" in h:
        adc = data[:, col("adc_u8")].astype(int)
        return time_s, adc, None, "raw"
    raise ValueError(f"Cabecera no reconocida: {header}")


# Busca captura_scope.csv (desde hantek/ o desde hantek/notebooks/)
_here = Path.cwd()
for CSV in (_here / "captura_scope.csv", _here.parent / "captura_scope.csv"):
    if CSV.exists():
        break
else:
    raise FileNotFoundError(
        "No está captura_scope.csv. Generá uno: "
        "python -m hantek_usb.cli get-real-data --parse --export-csv captura_scope.csv "
        "--count-a 0x400 --count-b 0"
    )

time_s, a, b, mode = load_hantek_scope_csv(CSV)

if mode == "interleaved":
    print(f"Muestras por canal: {len(a)}  |  CH1 min={a.min()} max={a.max()}  |  CH2 min={b.min()} max={b.max()}")
else:
    print(f"Muestras: {len(a)}  |  ADC min={a.min()}  max={a.max()}")
print(f"CSV ({mode}): {CSV.resolve()}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
if mode == "interleaved":
    ax.plot(time_s, a, lw=0.8, color="C0", label="CH1")
    ax.plot(time_s, b, lw=0.8, color="C1", label="CH2")
    ax.legend(loc="upper right")
    ax.set_ylabel("adc_u8 (crudo, por canal)")
else:
    ax.plot(time_s, a, lw=0.8, color="C0")
    ax.set_ylabel("adc_u8 (crudo)")
ax.set_xlabel("time_s (por defecto 1 s entre muestras; ajustá --csv-dt en la captura)")
ax.set_title("Captura osciloscopio Hantek")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()